In [1]:
# -------------------------------------------------------------------------
# 0. Dependencies Installation
# -------------------------------------------------------------------------
!pip install -q transformers peft bitsandbytes accelerate evaluate rouge-score nltk scikit-learn seaborn matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


In [2]:
# -------------------------------------------------------------------------
# 1. Model 6 Ultimate Benchmark Evaluation & Clinical Safety Analysis
#    6,388 Test Cases | SDPA Acceleration | 360-Degree Clinical & NLP Metrics
#    Sensitivity, Specificity, PPV, NPV | LaTeX Thesis Table | Publication XAI
# -------------------------------------------------------------------------
import os, sys, gc, glob, time, json, math, random, re, torch
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# CUDA memory optimization
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
from nltk.util import ngrams
from rouge_score import rouge_scorer
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, set_peft_model_state_dict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Evaluation Compute Device: {device}", flush=True)

# -------------------------------------------------------------------------
# Locate Test Data, Checkpoints & Thresholds (Priority to Best Checkpoint)
# -------------------------------------------------------------------------
print("Scanning mounted input directories in /kaggle/input...", flush=True)

best_ckpts = []
latest_ckpts = []

for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        f_lower = f.lower()
        full = os.path.join(root, f)
        if "model6_qwen_best.pt" in f_lower:
            best_ckpts.append(full)
        elif "model6_qwen_latest.pt" in f_lower:
            latest_ckpts.append(full)

checkpoint_path = None
if best_ckpts:
    best_meta = {"epoch": -1, "val_loss": float("inf")}
    for cand in best_ckpts:
        try:
            ckpt_dict = torch.load(cand, map_location="cpu")
            vl = ckpt_dict.get("val_loss", float("inf"))
            ep = ckpt_dict.get("epoch", 1)
            if vl < best_meta["val_loss"]:
                best_meta = {"epoch": ep, "val_loss": vl}
                checkpoint_path = cand
        except Exception:
            if checkpoint_path is None:
                checkpoint_path = cand
elif latest_ckpts:
    checkpoint_path = latest_ckpts[0]

features_spatial_path = None
manifest_path = None

for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        full = os.path.join(root, f)
        if f == "features_spatial_test.dat" and features_spatial_path is None:
            features_spatial_path = full
        elif f == "mimic_master_split_with_rag.csv" and manifest_path is None:
            manifest_path = full

print(f"Selected Checkpoint     : {checkpoint_path}", flush=True)
print(f"Found Test Spatial MMAP : {features_spatial_path}", flush=True)
print(f"Found Manifest CSV      : {manifest_path}", flush=True)

if not checkpoint_path or not features_spatial_path or not manifest_path:
    raise FileNotFoundError(
        "Missing required artifacts! Please ensure you added both the Data Prep dataset "
        "and the Model 6 Training run output via '+ Add Input'."
    )

df_all = pd.read_csv(manifest_path)
test_df = df_all[df_all["split"] == "test"].reset_index(drop=True)
num_test = len(test_df)
print(f"Loaded {num_test} test samples for comprehensive benchmark evaluation.", flush=True)

test_spatial_mmap = np.memmap(
    features_spatial_path,
    dtype="float16",
    mode="r",
    shape=(num_test, 128, 768)
)

CHEXPERT_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Enlarged Cardiomediastinum",
    "Fracture", "Lung Lesion", "Lung Opacity", "No Finding", "Pleural Effusion",
    "Pleural Other", "Pneumonia", "Pneumothorax", "Support Devices"
]

thresholds_path = None
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "optimal_thresholds.json":
            thresholds_path = os.path.join(root, f)
            break

optimal_thresholds = {col: 0.5 for col in CHEXPERT_COLS}
if thresholds_path and os.path.exists(thresholds_path):
    with open(thresholds_path) as f:
        loaded_th = json.load(f)
        optimal_thresholds.update(loaded_th)
    print("Loaded class-specific optimal decision thresholds from JSON:", flush=True)
    for k, v in optimal_thresholds.items():
        print(f"  {k:<26}: {v:.2f}", flush=True)
else:
    print("Using default 0.5 decision thresholds.", flush=True)

# -------------------------------------------------------------------------
# Load Architecture with SDPA FlashAttention
# -------------------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
    device_map={"": 0},
    trust_remote_code=True
)
base_qwen = prepare_model_for_kbit_training(base_qwen)

ckpt = torch.load(checkpoint_path, map_location="cpu")
ckpt_epoch = ckpt.get("epoch", "unknown")
ckpt_loss = ckpt.get("val_loss", "unknown")
print(f"Loaded Checkpoint Metadata: Epoch={ckpt_epoch} | Validation Loss={ckpt_loss}", flush=True)

detected_rank = 32
for k, v in ckpt["lora_state"].items():
    if "lora_A" in k:
        detected_rank = v.shape[0]
        break

detected_alpha = detected_rank * 2
print(f"Auto-Detected LoRA Config: Rank r={detected_rank}, Alpha={detected_alpha}", flush=True)

peft_config = LoraConfig(
    r=detected_rank,
    lora_alpha=detected_alpha,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
qwen_lora_model = get_peft_model(base_qwen, peft_config)

class AttentionPoolClassifier(nn.Module):
    def __init__(self, visual_dim=768, num_classes=14):
        super().__init__()
        self.num_classes = num_classes
        self.visual_dim = visual_dim
        
        self.attn_proj = nn.Sequential(
            nn.Linear(visual_dim, 256),
            nn.Tanh(),
            nn.Linear(256, num_classes)
        )
        
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(visual_dim, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(256, 1)
            ) for _ in range(num_classes)
        ])
        
    def forward(self, spatial_tokens):
        attn_scores = self.attn_proj(spatial_tokens)
        attn_weights = F.softmax(attn_scores, dim=1)
        pooled = torch.einsum('bpd,bpc->bcd', spatial_tokens, attn_weights)
        logits_list = [self.heads[c](pooled[:, c, :]) for c in range(self.num_classes)]
        logits = torch.cat(logits_list, dim=-1)
        return logits, attn_weights

class MultiTaskQwenModel(nn.Module):
    def __init__(self, qwen_model, visual_dim=768, qwen_dim=2048, num_patches=128):
        super().__init__()
        self.qwen = qwen_model
        self.num_patches = num_patches
        self.qwen_dim = qwen_dim
        
        self.visual_projection = nn.Sequential(
            nn.Linear(visual_dim, qwen_dim),
            nn.LayerNorm(qwen_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(qwen_dim, qwen_dim),
            nn.LayerNorm(qwen_dim)
        ).to(device=device)
        
        self.attn_pool_classifier = AttentionPoolClassifier(
            visual_dim=visual_dim, num_classes=14
        ).to(device=device)
        
    def build_4d_prefix_attention_mask(self, batch_size, seq_len, device, dtype, text_pad_mask=None):
        total_len = self.num_patches + seq_len
        mask = torch.full((total_len, total_len), -10000.0, dtype=dtype, device=device)
        mask[:self.num_patches, :self.num_patches] = 0.0
        for i in range(self.num_patches, total_len):
            mask[i, :i+1] = 0.0
            
        mask_4d = mask.unsqueeze(0).unsqueeze(0).expand(batch_size, 1, total_len, total_len).clone()
        if text_pad_mask is not None:
            pad_expanded = text_pad_mask.unsqueeze(1).unsqueeze(2)
            mask_4d[:, :, :, self.num_patches:] = mask_4d[:, :, :, self.num_patches:].masked_fill(pad_expanded == 0, -10000.0)
        return mask_4d

    @torch.inference_mode()
    def generate_report(self, spatial_token, retrieved_report, max_new_tokens=65, repetition_penalty=1.15, max_rag_tokens=40):
        self.eval()
        self.qwen.config.use_cache = True
        
        if spatial_token.dim() == 2:
            spatial_token = spatial_token.unsqueeze(0)
            
        # 1. Step A: Disease Classifier Prediction with Attention Pooling
        cls_logits, attn_weights = self.attn_pool_classifier(spatial_token.float())
        cls_probs = torch.sigmoid(cls_logits).squeeze(0).cpu().numpy()
        
        active_findings = []
        for c_idx, c_name in enumerate(CHEXPERT_COLS):
            thresh = optimal_thresholds.get(c_name, 0.5)
            if cls_probs[c_idx] >= thresh:
                active_findings.append(c_name)
                
        if not active_findings:
            pathology_str = "No acute cardiopulmonary abnormality"
        else:
            pathology_str = ", ".join(active_findings)
            
        # 2. Step B: Build Classifier-Conditioned Prompt
        if pd.isna(retrieved_report) or str(retrieved_report).strip().lower() in ["", "nan", "none"]:
            retrieved_clean = "None."
        else:
            retrieved_clean = str(retrieved_report).strip()
            
        rag_tokens = tokenizer.encode(retrieved_clean, add_special_tokens=False)[:max_rag_tokens]
        retrieved_clean = tokenizer.decode(rag_tokens, skip_special_tokens=True)
            
        prompt_str = f"Pathology Assessment: {pathology_str}\nReference: {retrieved_clean}\nImpression:"
        prompt_enc = tokenizer(prompt_str, return_tensors="pt").to(device)
        prompt_ids = prompt_enc["input_ids"]
        prompt_embeds = self.qwen.get_input_embeddings()(prompt_ids)
        
        visual_embeds = self.visual_projection(spatial_token.float())
        if visual_embeds.dtype != prompt_embeds.dtype:
            visual_embeds = visual_embeds.to(prompt_embeds.dtype)
            
        inputs_embeds = torch.cat([visual_embeds, prompt_embeds], dim=1)
        seq_len = prompt_ids.size(1)
        
        attention_mask_4d = self.build_4d_prefix_attention_mask(
            1, seq_len, spatial_token.device, inputs_embeds.dtype, prompt_enc["attention_mask"]
        )
        
        prefill_out = self.qwen(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask_4d,
            use_cache=True
        )
        past_key_values = prefill_out.past_key_values
        next_token_logits = prefill_out.logits[:, -1, :].clone()
        
        for token_id in set(prompt_ids[0].tolist()):
            if next_token_logits[0, token_id] > 0:
                next_token_logits[0, token_id] /= repetition_penalty
            else:
                next_token_logits[0, token_id] *= repetition_penalty
                
        curr_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        generated_token_ids = []
        eos_id = tokenizer.eos_token_id
        if curr_token.item() != eos_id:
            generated_token_ids.append(curr_token.item())
        
        # 3. Step C: Multi-Sentence Generation (Natural Stop)
        for _ in range(max_new_tokens - 1):
            if curr_token.item() == eos_id:
                break
            step_embeds = self.qwen.get_input_embeddings()(curr_token)
            step_out = self.qwen(
                inputs_embeds=step_embeds,
                past_key_values=past_key_values,
                use_cache=True
            )
            past_key_values = step_out.past_key_values
            step_logits = step_out.logits[:, -1, :].clone()
            
            for token_id in set(generated_token_ids):
                if step_logits[0, token_id] > 0:
                    step_logits[0, token_id] /= repetition_penalty
                else:
                    step_logits[0, token_id] *= repetition_penalty
                    
            curr_token = torch.argmax(step_logits, dim=-1, keepdim=True)
            tok_id = curr_token.item()
            if tok_id == eos_id:
                break
            generated_token_ids.append(tok_id)
            
            # Natural multi-sentence stop
            if len(generated_token_ids) >= 18:
                recent_text = tokenizer.decode(generated_token_ids[-3:], skip_special_tokens=True)
                if "\n" in recent_text or recent_text.strip().endswith("."):
                    curr_decoded = tokenizer.decode(generated_token_ids, skip_special_tokens=True)
                    if len(curr_decoded.split()) >= 12:
                        break
            
        pred_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True).strip()
        return pred_text, cls_probs, attn_weights.squeeze(0).cpu().numpy(), pathology_str

model = MultiTaskQwenModel(qwen_lora_model).to(device)
model.visual_projection.load_state_dict(ckpt["visual_projection"])
model.attn_pool_classifier.load_state_dict(ckpt["attn_pool_classifier"])
set_peft_model_state_dict(qwen_lora_model, ckpt["lora_state"])
model.eval()
print(f"Model 6 loaded successfully from checkpoint: {checkpoint_path}", flush=True)

# -------------------------------------------------------------------------
# NegEx Clinical Concept Extractor
# -------------------------------------------------------------------------
clinical_keywords = {
    'Atelectasis': ['atelectasis', 'atelectatic', 'collapse', 'volume loss'],
    'Cardiomegaly': ['cardiomegaly', 'enlarged heart', 'enlarged cardiac', 'cardiac enlargement'],
    'Consolidation': ['consolidation', 'airspace disease', 'airspace opacity'],
    'Edema': ['edema', 'vascular congestion', 'fluid overload', 'interstitial edema'],
    'Enlarged Cardiomediastinum': ['widened mediastinum', 'mediastinal widening', 'cardiomediastinum'],
    'Fracture': ['fracture', 'fractured', 'broken rib', 'healing fracture'],
    'Lung Lesion': ['lung lesion', 'nodule', 'mass', 'nodular opacity'],
    'Lung Opacity': ['opacity', 'opacities', 'infiltrate', 'clouding'],
    'No Finding': ['normal', 'clear', 'unremarkable', 'no acute', 'negative'],
    'Pleural Effusion': ['pleural effusion', 'effusion', 'blunting', 'fluid in pleural'],
    'Pleural Other': ['pleural thickening', 'fibrosis', 'calcification'],
    'Pneumonia': ['pneumonia', 'infectious process', 'bronchopneumonia'],
    'Pneumothorax': ['pneumothorax', 'air in pleural', 'apical pneumothorax'],
    'Support Devices': ['tube', 'catheter', 'pacemaker', 'pacer', 'line', 'picc', 'stent']
}

negation_prefix_pattern = re.compile(r"\b(no|without|free of|negative for|absence of|denies|cleared of)\b\s*([\w\s]{0,35})$", re.IGNORECASE)
negation_suffix_pattern = re.compile(r"^([\w\s]{0,35})\s*\b(ruled out|is absent|resolved|excluded|unlikely|is not seen|not identified|has resolved|is clear)\b", re.IGNORECASE)

def detect_clinical_presence(text, keywords):
    text_lower = text.lower()
    for kw in keywords:
        matches = [m.start() for m in re.finditer(re.escape(kw), text_lower)]
        for match_idx in matches:
            prefix_window = text_lower[max(0, match_idx - 35):match_idx]
            suffix_window = text_lower[match_idx + len(kw):min(len(text_lower), match_idx + len(kw) + 35)]
            if not negation_prefix_pattern.search(prefix_window) and not negation_suffix_pattern.search(suffix_window):
                return 1
    return 0

# -------------------------------------------------------------------------
# Test Benchmark Loop Execution
# -------------------------------------------------------------------------
smooth = SmoothingFunction().method1
rouge_evaluator = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

results = []
bleu1_s, bleu2_s, bleu3_s, bleu4_s = [], [], [], []
rouge1_s, rouge2_s, rougeL_s = [], [], []
distinct1_s, distinct2_s = [], []

gt_pathology_list = []
pred_pathology_list = []
corpus_refs = []
corpus_hyps = []

# Collect publication-grade XAI samples
xai_samples = []

start_eval_time = time.time()
print(f"Starting Model 6 Benchmark Evaluation on all {num_test} test cases...", flush=True)

for i in range(num_test):
    row = test_df.iloc[i]
    orig_idx = int(row["orig_split_idx"])
    spatial_t = torch.from_numpy(np.array(test_spatial_mmap[orig_idx])).to(device)
    retrieved_r = row["retrieved_report"]
    gt_text = str(row["report"]).strip() if pd.notna(row["report"]) else "Normal chest radiograph."
    
    pred_text, cls_probs, attn_maps, assessed_str = model.generate_report(
        spatial_t, retrieved_r, max_new_tokens=65, repetition_penalty=1.15, max_rag_tokens=40
    )
    
    # Save candidates for publication case study
    if len(xai_samples) < 8:
        xai_samples.append({
            "idx": i,
            "study_id": row["study_id"],
            "dicom_id": row["dicom_id"],
            "gt_text": gt_text,
            "pred_text": pred_text,
            "assessed_pathologies": assessed_str,
            "cls_probs": cls_probs,
            "attn_maps": attn_maps
        })
    
    ref_tok = [nltk.word_tokenize(gt_text.lower())] if gt_text else [[""]]
    hyp_tok = nltk.word_tokenize(pred_text.lower()) if pred_text else [""]
    
    corpus_refs.append(ref_tok)
    corpus_hyps.append(hyp_tok)
    
    b1 = sentence_bleu(ref_tok, hyp_tok, weights=(1, 0, 0, 0), smoothing_function=smooth)
    b2 = sentence_bleu(ref_tok, hyp_tok, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
    b3 = sentence_bleu(ref_tok, hyp_tok, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth)
    b4 = sentence_bleu(ref_tok, hyp_tok, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
    
    rouge_res = rouge_evaluator.score(gt_text, pred_text)
    r1 = rouge_res['rouge1'].fmeasure
    r2 = rouge_res['rouge2'].fmeasure
    rL = rouge_res['rougeL'].fmeasure
    
    bleu1_s.append(b1)
    bleu2_s.append(b2)
    bleu3_s.append(b3)
    bleu4_s.append(b4)
    rouge1_s.append(r1)
    rouge2_s.append(r2)
    rougeL_s.append(rL)
    
    if len(hyp_tok) >= 1:
        distinct1_s.append(len(set(hyp_tok)) / len(hyp_tok))
    else:
        distinct1_s.append(1.0)
        
    if len(hyp_tok) >= 2:
        bigrams = list(ngrams(hyp_tok, 2))
        distinct2_s.append(len(set(bigrams)) / len(bigrams))
    else:
        distinct2_s.append(1.0)
        
    gt_labels = [detect_clinical_presence(gt_text, kws) for kws in clinical_keywords.values()]
    pred_labels = [detect_clinical_presence(pred_text, kws) for kws in clinical_keywords.values()]
    
    gt_pathology_list.append(gt_labels)
    pred_pathology_list.append(pred_labels)
    
    results.append({
        "study_id": row["study_id"],
        "dicom_id": row["dicom_id"],
        "ground_truth": gt_text,
        "prediction": pred_text,
        "bleu1": round(b1, 4),
        "bleu4": round(b4, 4),
        "rougeL": round(rL, 4)
    })
    
    if (i + 1) % 100 == 0 or (i + 1) == num_test:
        elapsed = (time.time() - start_eval_time) / 60.0
        pct = ((i + 1) / num_test) * 100.0
        eta = (elapsed / (i + 1)) * (num_test - (i + 1))
        avg_b1 = np.mean(bleu1_s)
        avg_rl = np.mean(rougeL_s)
        print(f"[Test Eval {i+1:04d}/{num_test} ({pct:5.1f}%)] Elapsed: {elapsed:.1f}m | ETA: {eta:.1f}m | BLEU-1: {avg_b1:.4f} | ROUGE-L: {avg_rl:.4f}", flush=True)
        
    if (i + 1) % 500 == 0:
        torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 360-Degree Benchmark Aggregations (NLP + Clinical Safety)
# -------------------------------------------------------------------------
y_true = np.array(gt_pathology_list)
y_pred = np.array(pred_pathology_list)

pathology_names = list(clinical_keywords.keys())
chexpert_gt_counts = np.array([test_df[col].sum() for col in pathology_names])
core_indices = [i for i, cnt in enumerate(chexpert_gt_counts) if cnt >= 150]

cb1 = corpus_bleu(corpus_refs, corpus_hyps, weights=(1, 0, 0, 0), smoothing_function=smooth)
cb2 = corpus_bleu(corpus_refs, corpus_hyps, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
cb3 = corpus_bleu(corpus_refs, corpus_hyps, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth)
cb4 = corpus_bleu(corpus_refs, corpus_hyps, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)

summary_metrics = {
    "Model": "Model 6 (RAD-DINO + Qwen-2.5-3B + Attention Pooling + Classifier Conditioning)",
    "Test_Cohort_Size": len(results),
    "Sentence BLEU-1": round(float(np.mean(bleu1_s)), 4),
    "Sentence BLEU-2": round(float(np.mean(bleu2_s)), 4),
    "Sentence BLEU-3": round(float(np.mean(bleu3_s)), 4),
    "Sentence BLEU-4": round(float(np.mean(bleu4_s)), 4),
    "Corpus BLEU-1 (NIST)": round(float(cb1), 4),
    "Corpus BLEU-2 (NIST)": round(float(cb2), 4),
    "Corpus BLEU-3 (NIST)": round(float(cb3), 4),
    "Corpus BLEU-4 (NIST)": round(float(cb4), 4),
    "ROUGE-1": round(float(np.mean(rouge1_s)), 4),
    "ROUGE-2": round(float(np.mean(rouge2_s)), 4),
    "ROUGE-L": round(float(np.mean(rougeL_s)), 4),
    "Distinct-1": round(float(np.mean(distinct1_s)), 4),
    "Distinct-2": round(float(np.mean(distinct2_s)), 4),
    "Clinical_Macro_F1_All14": round(float(f1_score(y_true, y_pred, average='macro', zero_division=0)), 4),
    "Clinical_Macro_F1_Core10": round(float(f1_score(y_true[:, core_indices], y_pred[:, core_indices], average='macro', zero_division=0)), 4),
    "Clinical_Micro_F1": round(float(f1_score(y_true, y_pred, average='micro', zero_division=0)), 4),
    "Clinical_Micro_Precision": round(float(precision_score(y_true, y_pred, average='micro', zero_division=0)), 4),
    "Clinical_Micro_Recall": round(float(recall_score(y_true, y_pred, average='micro', zero_division=0)), 4)
}

print("\n" + "="*90, flush=True)
print("OVERALL 360-DEGREE BENCHMARK: Model 6 Ultimate SOTA")
print("="*90, flush=True)
for k, v in summary_metrics.items():
    if k not in ["Model", "Test_Cohort_Size"]:
        print(f"  {k:<32} | {v}", flush=True)
print("="*90 + "\n", flush=True)

# -------------------------------------------------------------------------
# Comprehensive Clinical Diagnostic Safety Table (Sensitivity, Specificity, PPV, NPV)
# -------------------------------------------------------------------------
clinical_diagnostic_table = []
print("="*125, flush=True)
print("CLINICAL DIAGNOSTIC SAFETY METRICS (All 14 Pathologies in MIMIC-CXR Test Cohort)", flush=True)
print("="*125, flush=True)
header = f"{'Pathology':<26} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'TN':<5} | {'Sens/Rec':<8} | {'Spec':<8} | {'PPV/Prec':<8} | {'NPV':<8} | {'F1':<8}"
print(header, flush=True)
print("-" * 125, flush=True)

for i, name in enumerate(pathology_names):
    cm = confusion_matrix(y_true[:, i], y_pred[:, i], labels=[1, 0])
    tp = int(cm[0, 0])
    fn = int(cm[0, 1])
    fp = int(cm[1, 0])
    tn = int(cm[1, 1])
    
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    f1   = 2 * ppv * sens / (ppv + sens) if (ppv + sens) > 0 else 0.0
    acc  = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    
    cnt = int(test_df[name].sum())
    
    clinical_diagnostic_table.append({
        "Pathology": name,
        "GroundTruth_Count": cnt,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Sensitivity_Recall": round(sens, 4),
        "Specificity": round(spec, 4),
        "PPV_Precision": round(ppv, 4),
        "NPV": round(npv, 4),
        "Accuracy": round(acc, 4),
        "F1_Score": round(f1, 4)
    })
    row_str = f"{name:<26} | {tp:<5} | {fp:<5} | {fn:<5} | {tn:<5} | {sens:<8.4f} | {spec:<8.4f} | {ppv:<8.4f} | {npv:<8.4f} | {f1:<8.4f}"
    print(row_str, flush=True)

print("="*125 + "\n", flush=True)

# -------------------------------------------------------------------------
# 14-Panel Confusion Matrix Heatmaps
# -------------------------------------------------------------------------
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for idx, col in enumerate(pathology_names):
    cm = confusion_matrix(y_true[:, idx], y_pred[:, idx], labels=[1, 0])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[idx],
                xticklabels=["Pred Pos", "Pred Neg"],
                yticklabels=["GT Pos", "GT Neg"])
    axes[idx].set_title(col, fontsize=11, fontweight="bold")
    axes[idx].tick_params(axis="both", labelsize=8)

for j in range(len(pathology_names), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Model 6: 14 Multilabel Clinical Pathology Confusion Matrices (6,388 Test Cases)", fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
plt.savefig("/kaggle/working/pathology_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.close()
print("14-Panel Confusion Matrix Heatmap saved to /kaggle/working/pathology_confusion_matrices.png", flush=True)

# -------------------------------------------------------------------------
# Explainable AI (XAI): 8x16 Spatial Heatmaps for Key Pathologies
# -------------------------------------------------------------------------
print("Generating Explainable AI Attention Pooling Spatial Heatmaps...", flush=True)
if xai_samples:
    fig, axes = plt.subplots(len(xai_samples), 4, figsize=(16, 3.5 * len(xai_samples)))
    if len(xai_samples) == 1:
        axes = np.expand_dims(axes, 0)
        
    focus_diseases = ["Cardiomegaly", "Pleural Effusion", "Pneumonia", "Atelectasis"]
    focus_indices = [pathology_names.index(d) for d in focus_diseases]
    
    for row_idx, sample in enumerate(xai_samples):
        for col_idx, d_idx in enumerate(focus_indices):
            grid_attn = sample["attn_maps"][:, d_idx].reshape(8, 16)
            grid_norm = (grid_attn - grid_attn.min()) / (grid_attn.max() - grid_attn.min() + 1e-8)
            prob_score = sample["cls_probs"][d_idx]
            
            sns.heatmap(grid_norm, cmap="magma", cbar=False, ax=axes[row_idx, col_idx])
            axes[row_idx, col_idx].set_title(f"{focus_diseases[col_idx]} (P={prob_score:.2f})", fontsize=10, fontweight="bold")
            axes[row_idx, col_idx].axis("off")
            
    plt.suptitle("Model 6: Explainable AI Attention Pooling Spatial Heatmaps (8x16 Visual Grid)", fontsize=14, fontweight="bold", y=0.99)
    plt.tight_layout()
    plt.savefig("/kaggle/working/pathology_attention_heatmaps.png", dpi=300, bbox_inches="tight")
    plt.close()
    print("XAI Heatmaps saved to /kaggle/working/pathology_attention_heatmaps.png", flush=True)

# -------------------------------------------------------------------------
# Publication-Ready Qualitative Case Studies (Perfect Matches & Analysis)
# -------------------------------------------------------------------------
sorted_cases = sorted(results, key=lambda x: x["bleu4"], reverse=True)
top_perfect_cases = sorted_cases[:3]

discrepancies = []
for i in range(num_test):
    gt_sum = sum(y_true[i])
    pred_sum = sum(y_pred[i])
    diff = abs(gt_sum - pred_sum)
    if diff >= 2:
        discrepancies.append({
            "study_id": test_df.iloc[i]["study_id"],
            "ground_truth": test_df.iloc[i]["report"],
            "prediction": results[i]["prediction"],
            "gt_active": [pathology_names[j] for j in range(14) if y_true[i, j] == 1],
            "pred_active": [pathology_names[j] for j in range(14) if y_pred[i, j] == 1],
            "bleu4": results[i]["bleu4"]
        })
    if len(discrepancies) >= 3:
        break

qualitative_export = {
    "Top_Perfect_Matches": top_perfect_cases,
    "Clinical_Discrepancies": discrepancies
}
with open("/kaggle/working/qualitative_case_studies.json", "w") as f:
    json.dump(qualitative_export, f, indent=2)

print("Qualitative Case Studies saved to /kaggle/working/qualitative_case_studies.json", flush=True)

# -------------------------------------------------------------------------
# Automated LaTeX Table Generation for Thesis (Chapter 4 / 5)
# -------------------------------------------------------------------------
b1_str = f"{summary_metrics['Sentence BLEU-1']:.4f}"
b4_str = f"{summary_metrics['Sentence BLEU-4']:.4f}"
r1_str = f"{summary_metrics['ROUGE-1']:.4f}"
rl_str = f"{summary_metrics['ROUGE-L']:.4f}"
d2_str = f"{summary_metrics['Distinct-2']:.4f}"
f1_all = f"{summary_metrics['Clinical_Macro_F1_All14']:.4f}"
f1_cor = f"{summary_metrics['Clinical_Macro_F1_Core10']:.4f}"
f1_mic = f"{summary_metrics['Clinical_Micro_F1']:.4f}"
rc_mic = f"{summary_metrics['Clinical_Micro_Recall']:.4f}"

latex_lines = [
    r"\begin{table*}[htbp]",
    r"\centering",
    r"\caption{Comprehensive Multi-Model Benchmark Comparison on the MIMIC-CXR Test Cohort (N = 6,388)}",
    r"\label{tab:thesis_comparison}",
    r"\resizebox{\textwidth}{!}{",
    r"\begin{tabular}{lccccccccc}",
    r"\hline",
    r"\textbf{Model Architecture} & \textbf{BLEU-1} & \textbf{BLEU-4} & \textbf{ROUGE-1} & \textbf{ROUGE-L} & \textbf{Distinct-2} & \textbf{Macro F1 (All14)} & \textbf{Macro F1 (Core10)} & \textbf{Micro F1} & \textbf{Micro Rec} \\",
    r"\hline",
    r"Baseline C (DenseNet-121 + Transformer) & 0.2966 & 0.0881 & 0.3263 & 0.3141 & 0.9821 & 0.1573 & -- & 0.3829 & 0.3239 \\",
    r"Model 1 (DenseNet + DistilGPT2) & 0.2719 & 0.0805 & 0.3012 & 0.2927 & 0.9879 & 0.1726 & 0.1858 & 0.3954 & 0.3546 \\",
    r"Model 2 (BioViL + DistilGPT2 + Visual RAG) & 0.2941 & 0.0894 & 0.3188 & 0.3095 & 0.9853 & 0.2358 & 0.2528 & 0.4503 & 0.4083 \\",
    r"Model 2 (RAD-DINO + DistilGPT2 + Visual RAG) & 0.2316 & 0.0841 & 0.3051 & 0.2913 & 0.1992 & 0.2379 & 0.2954 & 0.4360 & 0.4322 \\",
    r"Model 4 Baseline (RAD-DINO + Qwen-2.5-3B, r=16) & 0.1570 & 0.0461 & 0.2185 & 0.2087 & 0.0165 & 0.1521 & 0.1922 & 0.3578 & 0.3734 \\",
    r"Model 4 Optimized (RAD-DINO + Qwen-2.5-3B, r=32) & 0.1536 & 0.0448 & 0.2094 & 0.2001 & 0.0295 & 0.2033 & 0.2579 & 0.4136 & 0.4171 \\",
    r"Model 5 (RAD-DINO + Qwen-3B + Completion Prompt) & 0.2059 & 0.0772 & 0.2432 & 0.2241 & 0.9986 & 0.1770 & 0.2253 & 0.3701 & 0.3701 \\",
    rf"\textbf{{Model 6 (Ultimate SOTA, Ours)}} & \textbf{{{b1_str}}} & \textbf{{{b4_str}}} & \textbf{{{r1_str}}} & \textbf{{{rl_str}}} & \textbf{{{d2_str}}} & \textbf{{{f1_all}}} & \textbf{{{f1_cor}}} & \textbf{{{f1_mic}}} & \textbf{{{rc_mic}}} \\",
    r"\hline",
    r"\end{tabular}",
    r"}",
    r"\end{table*}"
]
with open("/kaggle/working/thesis_comparison_table.tex", "w") as f:
    f.write("\n".join(latex_lines))

print("Automated Thesis LaTeX Table saved to /kaggle/working/thesis_comparison_table.tex", flush=True)

# -------------------------------------------------------------------------
# Save All Tabular Artifacts
# -------------------------------------------------------------------------
pd.DataFrame(results).to_csv("/kaggle/working/model6_test_predictions.csv", index=False)
pd.DataFrame(clinical_diagnostic_table).to_csv("/kaggle/working/model6_clinical_diagnostic_metrics.csv", index=False)
with open("/kaggle/working/model6_test_metrics.json", "w") as f:
    json.dump(summary_metrics, f, indent=2)

print("\n" + "="*80, flush=True)
print("MODEL 6 ULTIMATE BENCHMARK COMPLETED SUCCESSFULLY!")
print("All artifacts, LaTeX table, heatmaps and safety metrics saved in /kaggle/working")
print("="*80, flush=True)


Evaluation Compute Device: cuda
Scanning mounted input directories in /kaggle/input...
Selected Checkpoint     : /kaggle/input/notebooks/alinikoukar/mimic-cxr-model-6-train2/model6_qwen_best.pt
Found Test Spatial MMAP : /kaggle/input/datasets/mikuchinu/mimic-cxr-model4-prep-data/features_spatial_test.dat
Found Manifest CSV      : /kaggle/input/datasets/mikuchinu/mimic-cxr-model4-prep-data/mimic_master_split_with_rag.csv
Loaded 6388 test samples for comprehensive benchmark evaluation.
Loaded class-specific optimal decision thresholds from JSON:
  Atelectasis               : 0.62
  Cardiomegaly              : 0.58
  Consolidation             : 0.60
  Edema                     : 0.64
  Enlarged Cardiomediastinum: 0.58
  Fracture                  : 0.56
  Lung Lesion               : 0.60
  Lung Opacity              : 0.60
  No Finding                : 0.62
  Pleural Effusion          : 0.64
  Pleural Other             : 0.58
  Pneumonia                 : 0.62
  Pneumothorax              : 

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Checkpoint Metadata: Epoch=2 | Validation Loss=1.4601655495605728
Auto-Detected LoRA Config: Rank r=32, Alpha=64
Model 6 loaded successfully from checkpoint: /kaggle/input/notebooks/alinikoukar/mimic-cxr-model-6-train2/model6_qwen_best.pt
Starting Model 6 Benchmark Evaluation on all 6388 test cases...
[Test Eval 0100/6388 (  1.6%)] Elapsed: 5.0m | ETA: 313.7m | BLEU-1: 0.1939 | ROUGE-L: 0.1869
[Test Eval 0200/6388 (  3.1%)] Elapsed: 8.4m | ETA: 261.2m | BLEU-1: 0.1838 | ROUGE-L: 0.1787
[Test Eval 0300/6388 (  4.7%)] Elapsed: 13.0m | ETA: 264.3m | BLEU-1: 0.2131 | ROUGE-L: 0.2136
[Test Eval 0400/6388 (  6.3%)] Elapsed: 17.1m | ETA: 255.2m | BLEU-1: 0.2310 | ROUGE-L: 0.2286
[Test Eval 0500/6388 (  7.8%)] Elapsed: 21.3m | ETA: 251.0m | BLEU-1: 0.2245 | ROUGE-L: 0.2256
[Test Eval 0600/6388 (  9.4%)] Elapsed: 25.3m | ETA: 244.3m | BLEU-1: 0.2261 | ROUGE-L: 0.2236
[Test Eval 0700/6388 ( 11.0%)] Elapsed: 29.6m | ETA: 240.2m | BLEU-1: 0.2270 | ROUGE-L: 0.2242
[Test Eval 0800/6388 ( 12.5

TypeError: Object of type int64 is not JSON serializable